In [ ]:
%reload_ext autoreload
%autoreload 2

import os

import torch
import torch.nn as nn
import torchvision
from tqdm import tqdm

import pickle
from matplotlib import pyplot as plt

## 2. 迭代对抗训练

- 在上一部分中，你已经实现了第一个对抗防御算法；
- 接下来，请模仿**Two-Step对抗训练**算法实现**迭代对抗训练**算法，并测试其在训练集、测试集上的预测表现，以及其对FGSM、PGD的防御效果；

- 具体实验步骤如下：

  1. 将代码文件（Python文件与Notebook文件）上传到服务器端根目录；

  2. 将样本数据（Week567_img_label.pkl）上传至服务器端data/目录下；

  3. 将之前训练的模型参数（lenet5.pt）上传至服务器端model/目录下；

  4. 依照提示，完成**Notebook文件**中的TODO内容；

In [ ]:
from Week567_General_Code_Question import LeNet5, load_mnist, fgsm, pgd
from Week567_General_Code_Question import evaluate

In [ ]:
# Parameter
batch_size = 128
epsilon = 0.2
iter = 5
alpha = 0.07

In [ ]:
# Model
model = LeNet5()
model.load_state_dict(torch.load('model/lenet5.pt'))
model.eval()

# Data
criterion = nn.CrossEntropyLoss()
train_loader, test_loader = load_mnist(batch_size=batch_size)

### 实现迭代对抗训练
- 请在下面的block中分别实现基于FGSM和PGD的迭代对抗训练攻击函数：
  - adv_train_iter_fgsm(data_loader, epoch, lr, criterion, epsilon, adv_loss_weight=1.)
  - adv_train_iter_pgd(data_loader, epoch, lr, criterion, epsilon, iter=5, adv_loss_weight=1.)
- 算法流程
  1. 从data_loader中取出正常样本对(img,label)
  2. 使用之前实现的FGSM/PGD算法，基于(img,label)生成对抗样本(adv_img,label)
      > tips: 之前版本实现的FGSM/PGD算法最后包含了`.detach()`操作，因此梯度不会传递到adv_img上
  3. 基于正常样本和对抗样本分别计算loss然后求和，再反传梯度更新模型

- 基于FGSM迭代对抗训练cnn模型

In [ ]:
def adv_train_iter_fgsm(data_loader, epoch, lr, criterion, epsilon, adv_loss_weight=1.):
    model = LeNet5()
    model.load_state_dict(torch.load('model/lenet5.pt'))
    model.train()

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for e in range(epoch):
        t = tqdm(data_loader)
        for img, label in t:
            # TODO: 前向传播并对良性样本计算损失
            benign_loss = 0.

            # TODO: 生成对抗样本，然后进行前向传播并计算对抗样本的损失
            adv_img = None
            adv_loss = 0.
            
            # TODO: 计算总损失，然后反向传播
            loss = benign_loss + adv_loss_weight * adv_loss

            t.set_postfix(epoch=e, benign_loss=benign_loss.item(), adv_loss=adv_loss.item())

    return model

In [ ]:
lr = 0.01
epoch = 10
adv_loss_weight = 1.0

epsilon = 0.2 

cnn_fgsm_iter = adv_train_iter_fgsm(train_loader, epoch, lr, criterion, epsilon, adv_loss_weight)
torch.save(cnn_fgsm_iter.state_dict(), 'model/cnn_fgsm_iter.pt')

- 基于PGD迭代对抗训练cnn模型

In [ ]:
def adv_train_iter_pgd(data_loader, epoch, lr, criterion, epsilon, alpha, iter=5, adv_loss_weight=1.):
    model = LeNet5()
    model.load_state_dict(torch.load('model/lenet5.pt'))
    model.train()

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for e in range(epoch):
        t = tqdm(data_loader)
        for img, label in t:
            # TODO: 前向传播并对良性样本计算损失
            benign_loss = 0.

            # TODO: 生成对抗样本，然后进行前向传播并计算对抗样本的损失
            adv_img = None
            adv_loss = 0.
            
            # TODO: 计算总损失，然后反向传播
            loss = benign_loss + adv_loss_weight * adv_loss

            t.set_postfix(epoch=e, benign_loss=benign_loss.item(), adv_loss=adv_loss.item())

    return model

In [ ]:
lr = 0.01
epoch = 10
adv_loss_weight = 1.0

epsilon = 0.2
iter = 5

cnn_pgd_iter = adv_train_iter_pgd(train_loader, epoch, lr, criterion, epsilon, alpha, iter, adv_loss_weight)
torch.save(cnn_pgd_iter.state_dict(), 'model/cnn_pgd_iter.pt')
# 训练后保存好模型文件，以便检查时快速测试结果

### 评测模型性能

In [ ]:
from Week567_General_Code_Question import evaluate_dataloader

- 测试基于FGSM执行迭代对抗训练的CNN的预测质量

In [ ]:
evaluate_dataloader(test_loader, cnn_fgsm_iter)

- 测试基于PGD执行迭代对抗训练的CNN的预测质量

In [ ]:
evaluate_dataloader(test_loader, cnn_pgd_iter)

### 评测防御效果

In [ ]:
with open('data/Week567_img_label.pkl', 'rb') as f:
    data = pickle.load(f)
    imgs, labels = data['img'], data['label']

- 评测原始模型针对FGSM/PGD攻击的效果（作为参照）

In [ ]:
print("For Original Model.\n")
print("Against FGSM:")
epsilon = 0.2

adv_xs = fgsm(imgs, epsilon, model, criterion, labels)
pred_label = evaluate(adv_xs, labels, model)


print("Against PGD:")
alpha = 0.07
iter = 5

adv_xs = pgd(imgs, epsilon, alpha, iter, model, criterion, labels)

pred_label = evaluate(adv_xs, labels, model)

- 评测基于FGSM执行迭代对抗训练的模型针对FGSM/PGD攻击的防御效果

In [ ]:
print("For FGSM Iterative.\n")
print("Against FGSM:")
epsilon = 0.2

adv_xs = fgsm(imgs, epsilon, cnn_fgsm_iter, criterion, labels)
pred_label = evaluate(adv_xs, labels, cnn_fgsm_iter)


print("Against PGD:")
alpha = 0.07
iter = 5

adv_xs = pgd(imgs, epsilon, alpha, iter, cnn_fgsm_iter, criterion, labels)

pred_label = evaluate(adv_xs, labels, cnn_fgsm_iter)

- 评测基于PGD执行迭代对抗训练的模型针对FGSM/PGD攻击的防御效果

In [ ]:
print("For PGD Iterative.\n")
print("Against FGSM:")
epsilon = 0.2

adv_xs = fgsm(imgs, epsilon, cnn_pgd_iter, criterion, labels)
pred_label = evaluate(adv_xs, labels, cnn_pgd_iter)


print("Against PGD:")
alpha = 0.07
iter = 5

adv_xs = pgd(imgs, epsilon, alpha, iter, cnn_pgd_iter, criterion, labels)

pred_label = evaluate(adv_xs, labels, cnn_pgd_iter)